In [1]:
# ============================================================
# CELL 1: SETUP AND IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import joblib
import time
import gc
from pathlib import Path

LOCAL_DIR = Path("C:\\Users\\admin\\Documents\\Glacier Project")
REGIONS = ['R1a', 'R1b', 'R2', 'R3']
PIXEL_AREA_M2 = 100  # 10m × 10m pixel = 100 m²

EASD_FEATURES = ['elevation', 'aspect', 'slope', 'edge_distance']
N_PCS = 10
PC_FEATURES = [f'PC{i+1}' for i in range(N_PCS)]
ALL_FEATURES = EASD_FEATURES + PC_FEATURES

print('Setup complete.')

Setup complete.


In [6]:
# ============================================================
# CELL 2: LOAD DATA AND TRAINED MODELS
# 
# Loads the full 15.2M-pixel Peru-wide dataset and the trained 
# RF models from the previous notebook. The PCA + scaler are
# applied here to get PC scores attached to each pixel.
# ============================================================
t0 = time.time()

dfs = []
for region in REGIONS:
    df = pd.read_parquet(LOCAL_DIR / f'Merged\\{region.lower()}_combined.parquet')
    df['region'] = region
    dfs.append(df)
df_all = pd.concat(dfs, ignore_index=True)
print(f'Loaded {len(df_all):,} pixels in {time.time()-t0:.1f}s')

# Load PCA artifacts
pca = joblib.load(LOCAL_DIR / 'Saved_Models\\pca_alphaearth.joblib')
scaler = joblib.load(LOCAL_DIR / 'Saved_Models\\scaler_alphaearth.joblib')

# Load trained RF models
rf_easd = joblib.load(LOCAL_DIR / 'Saved_Models\\rf_easd_new_baseline_v2.joblib')
rf_pc10 = joblib.load(LOCAL_DIR / 'Saved_Models\\rf_easd_pc10_v2.joblib')
print('Loaded PCA, scaler, and both RF models.')

Loaded 15,242,639 pixels in 6.6s
Loaded PCA, scaler, and both RF models.


In [7]:
# ============================================================
# CELL 3 (REVISED): PROJECT TO PCA IN-PLACE, MEMORY-AWARE
# ============================================================
import numpy as np
import gc

ae_cols = [c for c in df_all.columns 
           if c.startswith('A') and len(c) == 3 and c[1:].isdigit()]
assert len(ae_cols) == 64

# Process AE columns in-place: scale, transform, attach PCs, drop AE columns
# Use a temporary variable that we delete immediately
print('Standardising AE bands...')
ae_array = df_all[ae_cols].values.astype(np.float32)  # 3.6 GB
print(f'AE array: {ae_array.nbytes / 1e9:.2f} GB')

ae_array = scaler.transform(ae_array)
print('Scaled.')

ae_array = pca.transform(ae_array)[:, :N_PCS]  # 15M × 10 = ~600 MB, much smaller
print(f'PC array: {ae_array.nbytes / 1e9:.2f} GB')

# Attach PC columns to df
df_all[PC_FEATURES] = ae_array
del ae_array
gc.collect()

# Drop the now-unused AE columns to free 3.6 GB
df_all = df_all.drop(columns=ae_cols)
gc.collect()

print(f'After cleanup memory: {df_all.memory_usage(deep=True).sum() / 1e9:.2f} GB')

Standardising AE bands...
AE array: 3.90 GB
Scaled.
PC array: 0.61 GB
After cleanup memory: 2.00 GB


In [8]:
# ============================================================
# CELL 4 (REVISED): PREDICT IN CHUNKS, ONE MODEL AT A TIME
# ============================================================
def predict_in_chunks(model, X, chunk_size=500_000):
    """Predict melt probability in chunks to avoid materialising 
    a large probabilities array all at once."""
    n = len(X)
    probs = np.zeros(n, dtype=np.float32)
    for i in range(0, n, chunk_size):
        end = min(i + chunk_size, n)
        probs[i:end] = model.predict_proba(X[i:end])[:, 1]
        if (i // chunk_size) % 5 == 0:
            print(f'  {end:,} / {n:,} done')
    return probs

# === EASD-only first ===
print('=== EASD-only predictions ===')
t0 = time.time()
X_easd = df_all[EASD_FEATURES].values  # ~480 MB, fine
df_all['prob_easd'] = predict_in_chunks(rf_easd, X_easd, chunk_size=500_000)
del X_easd
gc.collect()
print(f'EASD-only done in {time.time()-t0:.1f}s')

# Free the EASD model now that predictions are stored
del rf_easd
gc.collect()
print('Freed EASD model.')

# Reload PC10 model fresh
rf_pc10 = joblib.load(LOCAL_DIR / 'Saved_Models/rf_easd_pc10_v2.joblib')

# === EASD+PC10 second ===
print('\n=== EASD+PC10 predictions ===')
t0 = time.time()
X_pc10 = df_all[ALL_FEATURES].values  # ~840 MB
df_all['prob_pc10'] = predict_in_chunks(rf_pc10, X_pc10, chunk_size=500_000)
del X_pc10
gc.collect()
print(f'EASD+PC10 done in {time.time()-t0:.1f}s')

del rf_pc10
gc.collect()

# Sanity check
print(f'\nPrediction summary:')
print(f'  EASD prob mean: {df_all["prob_easd"].mean():.3f}, '
      f'median: {df_all["prob_easd"].median():.3f}')
print(f'  PC10 prob mean: {df_all["prob_pc10"].mean():.3f}, '
      f'median: {df_all["prob_pc10"].median():.3f}')
print(f'  Actual melt rate: {df_all["melt_label"].mean():.3f}')

=== EASD-only predictions ===
  500,000 / 15,242,639 done
  3,000,000 / 15,242,639 done
  5,500,000 / 15,242,639 done
  8,000,000 / 15,242,639 done
  10,500,000 / 15,242,639 done
  13,000,000 / 15,242,639 done
  15,242,639 / 15,242,639 done
EASD-only done in 224.5s
Freed EASD model.

=== EASD+PC10 predictions ===
  500,000 / 15,242,639 done
  3,000,000 / 15,242,639 done
  5,500,000 / 15,242,639 done
  8,000,000 / 15,242,639 done
  10,500,000 / 15,242,639 done
  13,000,000 / 15,242,639 done
  15,242,639 / 15,242,639 done
EASD+PC10 done in 190.5s

Prediction summary:
  EASD prob mean: 0.318, median: 0.223
  PC10 prob mean: 0.283, median: 0.153
  Actual melt rate: 0.195


In [9]:
# ============================================================
# CELL 5: COMPUTE OBSERVED TOTAL MELT AREA (THE THRESHOLDING TARGET)
#
# The thresholded prediction works as follows:
#   1. Count actual melt pixels and convert to area (km²).
#   2. Rank all pixels by predicted melt probability (descending).
#   3. Take the top-N pixels whose total area equals observed melt.
#   4. Those top-N pixels are the model's "predicted melt set".
#   5. Compare to actual melt set via overlap metrics.
# 
# This step computes step 1.
# ============================================================
n_actual_melt = (df_all['melt_label'] == 1).sum()
n_total = len(df_all)

actual_melt_area_km2 = n_actual_melt * PIXEL_AREA_M2 / 1e6
total_ice_area_km2 = n_total * PIXEL_AREA_M2 / 1e6

print(f'=== Population stats ===')
print(f'  Total ice pixels:    {n_total:,}')
print(f'  Actual melt pixels:  {n_actual_melt:,}')
print(f'  Melt rate:           {n_actual_melt/n_total*100:.2f}%')
print(f'  Total ice area:      {total_ice_area_km2:.1f} km²')
print(f'  Actual melt area:    {actual_melt_area_km2:.1f} km²')
print(f'\n(For reference: Darina\'s reported 2016-2023 melt was 217.8 km².)')

=== Population stats ===
  Total ice pixels:    15,242,639
  Actual melt pixels:  2,971,684
  Melt rate:           19.50%
  Total ice area:      1524.3 km²
  Actual melt area:    297.2 km²

(For reference: Darina's reported 2016-2023 melt was 217.8 km².)


In [10]:
# ============================================================
# CELL 6: THRESHOLD AND COMPUTE OVERLAP - HELPER FUNCTION
# 
# Generic function: given pixel probabilities and actual labels,
# threshold to match observed melt area, then compute several 
# overlap metrics:
#   - Overlap %: (predicted ∩ actual) / actual  (i.e. recall)
#   - IoU:        (predicted ∩ actual) / (predicted ∪ actual)
#   - Precision:  (predicted ∩ actual) / predicted
#   - Threshold value used (probability cutoff)
# ============================================================
def thresholded_overlap(probs, labels, target_n_melt):
    """
    Threshold predictions to predict exactly target_n_melt pixels as melt.
    Compare against actual labels.
    
    probs: array of predicted melt probabilities, shape (N,)
    labels: array of true binary labels (0 = non-melt, 1 = melt), shape (N,)
    target_n_melt: integer, number of pixels to predict as melt
                   (typically equals actual count of melt pixels)
    """
    # Get indices of top-N most-vulnerable pixels
    threshold_value = np.partition(probs, -target_n_melt)[-target_n_melt]
    predicted_melt = probs >= threshold_value
    
    # In case of ties at the threshold, the actual count may exceed target.
    # Trim using argsort to get exactly target_n_melt predicted melts.
    if predicted_melt.sum() != target_n_melt:
        top_indices = np.argsort(probs)[::-1][:target_n_melt]
        predicted_melt = np.zeros(len(probs), dtype=bool)
        predicted_melt[top_indices] = True
    
    actual_melt = labels == 1
    
    n_predicted = predicted_melt.sum()
    n_actual = actual_melt.sum()
    n_intersect = (predicted_melt & actual_melt).sum()
    n_union = (predicted_melt | actual_melt).sum()
    
    overlap_pct = n_intersect / n_actual * 100  # equivalent to recall
    precision_pct = n_intersect / n_predicted * 100
    iou = n_intersect / n_union
    
    return {
        'threshold': threshold_value,
        'n_predicted': n_predicted,
        'n_actual': n_actual,
        'n_intersect': n_intersect,
        'overlap_pct': overlap_pct,
        'precision_pct': precision_pct,
        'iou': iou,
    }

# Sanity check the function
print('Helper function defined.')

Helper function defined.


In [11]:
# ============================================================
# CELL 7: COMPUTE OVERLAP FOR BOTH MODELS
# 
# Runs the thresholding + overlap computation for the baseline and
# the AlphaEarth-augmented model. Reports the comparison.
# ============================================================
print('=== EASD-only model ===')
result_easd = thresholded_overlap(
    df_all['prob_easd'].values,
    df_all['melt_label'].values,
    target_n_melt=n_actual_melt
)
for k, v in result_easd.items():
    if isinstance(v, float):
        print(f'  {k:15s} {v:.4f}')
    else:
        print(f'  {k:15s} {v:,}')

print('\n=== EASD + PC1-PC10 model ===')
result_pc10 = thresholded_overlap(
    df_all['prob_pc10'].values,
    df_all['melt_label'].values,
    target_n_melt=n_actual_melt
)
for k, v in result_pc10.items():
    if isinstance(v, float):
        print(f'  {k:15s} {v:.4f}')
    else:
        print(f'  {k:15s} {v:,}')

# Comparison summary
print('\n=== COMPARISON ===')
print(f'{"Metric":<25}{"EASD-only":<15}{"EASD+PC10":<15}{"Δ":<10}')
print(f'{"Overlap % (recall)":<25}'
      f'{result_easd["overlap_pct"]:<15.2f}'
      f'{result_pc10["overlap_pct"]:<15.2f}'
      f'{result_pc10["overlap_pct"]-result_easd["overlap_pct"]:+.2f}')
print(f'{"Precision %":<25}'
      f'{result_easd["precision_pct"]:<15.2f}'
      f'{result_pc10["precision_pct"]:<15.2f}'
      f'{result_pc10["precision_pct"]-result_easd["precision_pct"]:+.2f}')
print(f'{"IoU":<25}'
      f'{result_easd["iou"]:<15.4f}'
      f'{result_pc10["iou"]:<15.4f}'
      f'{result_pc10["iou"]-result_easd["iou"]:+.4f}')

print(f'\n(Reference: Darina\'s reported 2016-2023 overlap was 74.9%)')

=== EASD-only model ===
  threshold       0.6299999952316284
  n_predicted     2,971,684
  n_actual        2,971,684
  n_intersect     1,864,204
  overlap_pct     62.7322
  precision_pct   62.7322
  iou             0.4570

=== EASD + PC1-PC10 model ===
  threshold       0.6000000238418579
  n_predicted     2,971,684
  n_actual        2,971,684
  n_intersect     2,209,638
  overlap_pct     74.3564
  precision_pct   74.3564
  iou             0.5918

=== COMPARISON ===
Metric                   EASD-only      EASD+PC10      Δ         
Overlap % (recall)       62.73          74.36          +11.62
Precision %              62.73          74.36          +11.62
IoU                      0.4570         0.5918         +0.1348

(Reference: Darina's reported 2016-2023 overlap was 74.9%)


In [12]:
# ============================================================
# CELL 8: SAVE RESULTS
# 
# Save the per-pixel predictions for downstream analysis
# (per-glacier overlap, visualisation, etc.). The full df with
# probabilities is large but we only need a subset of columns.
# ============================================================
output_cols = ['region', 'lon', 'lat', 'melt_label', 'edge_distance',
               'prob_easd', 'prob_pc10']
df_predictions = df_all[output_cols]
df_predictions.to_parquet(LOCAL_DIR / 'Results\\predictions_full_peru_v2.parquet')
print(f'Saved predictions: {len(df_predictions):,} rows')
print(f'Columns: {df_predictions.columns.tolist()}')

Saved predictions: 15,242,639 rows
Columns: ['region', 'lon', 'lat', 'melt_label', 'edge_distance', 'prob_easd', 'prob_pc10']
